# Problem Set 6 - Solución computacional

Cada ejercicio se desarrolla con simulaciones y visualizaciones en Python, siguiendo el enfoque indicado en el documento de solución.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats, integrate, special
import matplotlib.pyplot as plt
from pathlib import Path

rng = np.random.default_rng(20260530)
plt.style.use("seaborn-v0_8-whitegrid")


## Problema 1

Generación de números aleatorios con inversión y aceptación-rechazo, con validación gráfica frente a las densidades teóricas.

In [ ]:
alphaValue = 1.5
betaValue = 1.0
rateValue = 1.0
sampleSize = 20000


def sampleWeibull(alphaValue, betaValue, size, rng):
    uniformSample = rng.random(size)
    return betaValue * (-np.log(uniformSample)) ** (1 / alphaValue)


def sampleExponential(rateValue, size, rng):
    uniformSample = rng.random(size)
    return -np.log(uniformSample) / rateValue


def sampleBeta31(size, rng):
    uniformSample = rng.random(size)
    return uniformSample ** (1 / 3)


def sampleStudentT10(size, rng):
    mConstant = stats.t.pdf(0, df=10) / stats.cauchy.pdf(0)
    accepted = []
    acceptedCount = 0
    totalProposals = 0
    while acceptedCount < size:
        batchSize = max(1000, size - acceptedCount)
        uniformOne = rng.random(batchSize)
        cauchySample = np.tan(np.pi * (uniformOne - 0.5))
        uniformTwo = rng.random(batchSize)
        acceptProb = stats.t.pdf(cauchySample, df=10) / (mConstant * stats.cauchy.pdf(cauchySample))
        acceptedBatch = cauchySample[uniformTwo <= acceptProb]
        accepted.append(acceptedBatch)
        acceptedCount += acceptedBatch.size
        totalProposals += batchSize
    sample = np.concatenate(accepted)[:size]
    acceptanceRate = size / totalProposals
    return sample, acceptanceRate


weibullSample = sampleWeibull(alphaValue, betaValue, sampleSize, rng)
exponentialSample = sampleExponential(rateValue, sampleSize, rng)
studentSample, acceptanceRate = sampleStudentT10(sampleSize, rng)
betaSample = sampleBeta31(sampleSize, rng)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

xWeibull = np.linspace(0, np.percentile(weibullSample, 99.5), 400)
axes[0, 0].hist(weibullSample, bins=60, density=True, alpha=0.6, label="Muestra")
axes[0, 0].plot(xWeibull, stats.weibull_min.pdf(xWeibull, c=alphaValue, scale=betaValue), color="black", label="PDF teórica")
axes[0, 0].set_title("Weibull (α=1.5, β=1)")
axes[0, 0].set_xlabel("x")
axes[0, 0].set_ylabel("densidad")
axes[0, 0].legend()

xExp = np.linspace(0, np.percentile(exponentialSample, 99.5), 400)
axes[0, 1].hist(exponentialSample, bins=60, density=True, alpha=0.6, label="Muestra")
axes[0, 1].plot(xExp, stats.expon.pdf(xExp, scale=1 / rateValue), color="black", label="PDF teórica")
axes[0, 1].set_title("Exponencial (λ=1)")
axes[0, 1].set_xlabel("x")
axes[0, 1].set_ylabel("densidad")
axes[0, 1].legend()

xStudent = np.linspace(-6, 6, 400)
axes[1, 0].hist(studentSample, bins=60, density=True, alpha=0.6, label="Muestra")
axes[1, 0].plot(xStudent, stats.t.pdf(xStudent, df=10), color="black", label="PDF teórica")
axes[1, 0].set_title("t de Student (ν=10)")
axes[1, 0].set_xlabel("x")
axes[1, 0].set_ylabel("densidad")
axes[1, 0].legend()

xBeta = np.linspace(0, 1, 400)
axes[1, 1].hist(betaSample, bins=60, density=True, alpha=0.6, label="Muestra")
axes[1, 1].plot(xBeta, stats.beta.pdf(xBeta, a=3, b=1), color="black", label="PDF teórica")
axes[1, 1].set_title("f(x)=3x² en (0,1)")
axes[1, 1].set_xlabel("x")
axes[1, 1].set_ylabel("densidad")
axes[1, 1].legend()

fig.suptitle("Generación de variables aleatorias")
plt.tight_layout()
print(f"Tasa de aceptación t10: {acceptanceRate:.3f}")


## Problema 2

Integración por Monte Carlo con límites elegidos para controlar la contribución de colas.

In [ ]:
epsilonValue = 1e-6
sampleSize = 200000


def monteCarloIntegral(function, a, b, sampleSize, rng):
    uniformSample = rng.random(sampleSize)
    xSample = a + (b - a) * uniformSample
    values = function(xSample)
    estimate = (b - a) * values.mean()
    standardError = (b - a) * values.std(ddof=1) / np.sqrt(sampleSize)
    return estimate, standardError


gammaUpper = stats.gamma.ppf(1 - epsilonValue, a=5, scale=1)
tailLimit = -np.log(epsilonValue)

functionList = [
    ("x^2", lambda x: x**2, 0.0, 1.0, 1 / 3),
    ("e^x", lambda x: np.exp(x), 0.0, 1.0, np.e - 1),
    ("x^4 e^{-x}", lambda x: x**4 * np.exp(-x), 0.0, gammaUpper, special.gamma(5)),
    ("e^{-x} / (1 + (x-1)^2)", lambda x: np.exp(-x) / (1 + (x - 1) ** 2), 0.0, tailLimit,
     integrate.quad(lambda x: np.exp(-x) / (1 + (x - 1) ** 2), 0, np.inf)[0])
]

rows = []
for name, function, a, b, reference in functionList:
    estimate, standardError = monteCarloIntegral(function, a, b, sampleSize, rng)
    rows.append({
        "funcion": name,
        "a": a,
        "b": b,
        "estimacion": estimate,
        "errorEstandar": standardError,
        "referencia": reference
    })

resultTable = pd.DataFrame(rows)
resultTable


## Problema 3

Desviación respecto a la normal mediante métricas de distancia y simulación Monte Carlo.

In [ ]:
def andersonDarlingNormal(sample):
    sampleSorted = np.sort(sample)
    n = sampleSorted.size
    cdfValues = stats.norm.cdf(sampleSorted)
    cdfValues = np.clip(cdfValues, 1e-12, 1 - 1e-12)
    index = np.arange(1, n + 1)
    term = (2 * index - 1) * (np.log(cdfValues) + np.log(1 - cdfValues[::-1]))
    statistic = -n - term.mean()
    return statistic


sampleSize = 100
tScale = np.sqrt(5 / 3)
altSample = stats.t.rvs(df=5, size=sampleSize, random_state=rng) / tScale

ksResult = stats.kstest(altSample, "norm")
cvmResult = stats.cramervonmises(altSample, "norm")
adStatistic = andersonDarlingNormal(altSample)

metricsTable = pd.DataFrame([
    {"metrica": "KS", "estadistico": ksResult.statistic, "valorP": ksResult.pvalue},
    {"metrica": "Cramér-von Mises", "estadistico": cvmResult.statistic, "valorP": cvmResult.pvalue},
    {"metrica": "Anderson-Darling", "estadistico": adStatistic, "valorP": np.nan}
])
metricsTable


In [ ]:
replications = 4000
adNull = np.empty(replications)
adAlt = np.empty(replications)

for index in range(replications):
    normalSample = rng.normal(0, 1, size=sampleSize)
    adNull[index] = andersonDarlingNormal(normalSample)
    altSampleLoop = stats.t.rvs(df=5, size=sampleSize, random_state=rng) / tScale
    adAlt[index] = andersonDarlingNormal(altSampleLoop)

threshold = np.quantile(adNull, 0.95)
powerEstimate = (adAlt > threshold).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(adNull, bins=40, density=True, alpha=0.6, label="H0: Normal")
axes[0].hist(adAlt, bins=40, density=True, alpha=0.6, label="H1: t5")
axes[0].axvline(threshold, color="black", linestyle="--", label="Umbral 95%")
axes[0].set_xlabel("Estadístico Anderson-Darling")
axes[0].set_ylabel("densidad")
axes[0].set_title(f"Potencia estimada: {powerEstimate:.3f}")
axes[0].legend()

stats.probplot(altSample, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot muestra t5 estandarizada")

plt.tight_layout()


## Problema 4

Prueba de hipótesis Monte Carlo para la media con varianza conocida y curva de potencia.

In [ ]:
def monteCarloZTest(sample, mu0, sigma, alpha, simulations, rng):
    n = len(sample)
    zObs = (np.mean(sample) - mu0) / (sigma / np.sqrt(n))
    simulated = rng.normal(mu0, sigma, size=(simulations, n))
    zSim = (simulated.mean(axis=1) - mu0) / (sigma / np.sqrt(n))
    pValue = (np.sum(np.abs(zSim) >= abs(zObs)) + 1) / (simulations + 1)
    decision = pValue <= alpha
    return zObs, pValue, decision, zSim


sampleSize = 30
sigmaValue = 1.0
mu0Value = 0.0
alphaValue = 0.05

observedSample = rng.normal(0.35, sigmaValue, size=sampleSize)
zObs, pValue, decision, zSim = monteCarloZTest(observedSample, mu0Value, sigmaValue, alphaValue, 20000, rng)

muGrid = np.linspace(-0.6, 0.6, 9)
replications = 4000
zCritical = stats.norm.ppf(1 - alphaValue / 2)

powerValues = []
for muAlt in muGrid:
    simulated = rng.normal(muAlt, sigmaValue, size=(replications, sampleSize))
    zAlt = (simulated.mean(axis=1) - mu0Value) / (sigmaValue / np.sqrt(sampleSize))
    powerValues.append((np.abs(zAlt) >= zCritical).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(zSim, bins=50, density=True, alpha=0.6, label="Distribución nula")
axes[0].axvline(zObs, color="black", linestyle="--", label=f"Z observado = {zObs:.2f}")
axes[0].axvline(-zObs, color="black", linestyle="--")
axes[0].set_xlabel("Z")
axes[0].set_ylabel("densidad")
axes[0].set_title(f"p-valor MC = {pValue:.4f} | Rechazo = {decision}")
axes[0].legend()

axes[1].plot(muGrid, powerValues, marker="o")
axes[1].set_xlabel("Media alternativa")
axes[1].set_ylabel("potencia")
axes[1].set_ylim(0, 1)
axes[1].set_title("Curva de potencia")

plt.tight_layout()


## Problema 5

Prueba de Monte Carlo para la duración de baterías y comparación con el valor teórico.

In [ ]:
batteryData = np.array([10.8, 11.5, 12.1, 10.9, 11.3, 11.0, 12.4, 10.7])
mu0Value = 12.0
sigmaValue = 1.5
alphaValue = 0.05

sampleSize = batteryData.size
zObs = (batteryData.mean() - mu0Value) / (sigmaValue / np.sqrt(sampleSize))

replications = 20000
simulated = rng.normal(mu0Value, sigmaValue, size=(replications, sampleSize))
zSim = (simulated.mean(axis=1) - mu0Value) / (sigmaValue / np.sqrt(sampleSize))

pValueMc = (np.sum(np.abs(zSim) >= abs(zObs)) + 1) / (replications + 1)
pValueTheo = 2 * (1 - stats.norm.cdf(abs(zObs)))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(zSim, bins=50, density=True, alpha=0.6, label="Distribución nula MC")
ax.axvline(zObs, color="black", linestyle="--", label=f"Z observado = {zObs:.2f}")
ax.axvline(-zObs, color="black", linestyle="--")
ax.set_xlabel("Z")
ax.set_ylabel("densidad")
ax.set_title(f"p-valor MC = {pValueMc:.4f} | p-valor teórico = {pValueTheo:.4f}")
ax.legend()


## Problema 6

Prueba chi-cuadrado para un dado justo con simulación Monte Carlo.

In [ ]:
observedCounts = np.array([4, 14, 10, 11, 7, 14])
faceCount = observedCounts.size
sampleSize = observedCounts.sum()
expectedCounts = np.full(faceCount, sampleSize / faceCount)

chiObs = ((observedCounts - expectedCounts) ** 2 / expectedCounts).sum()

replications = 20000
simulatedCounts = rng.multinomial(sampleSize, [1 / faceCount] * faceCount, size=replications)
chiSim = ((simulatedCounts - expectedCounts) ** 2 / expectedCounts).sum(axis=1)

pValueMc = (np.sum(chiSim >= chiObs) + 1) / (replications + 1)
pValueTheo = stats.chi2.sf(chiObs, df=faceCount - 1)

xGrid = np.linspace(0, max(chiSim.max(), chiObs) * 1.1, 400)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(chiSim, bins=50, density=True, alpha=0.6, label="MC")
ax.plot(xGrid, stats.chi2.pdf(xGrid, df=faceCount - 1), color="black", label="Chi-cuadrado teórica")
ax.axvline(chiObs, color="black", linestyle="--", label=f"χ² observado = {chiObs:.2f}")
ax.set_xlabel("χ²")
ax.set_ylabel("densidad")
ax.set_title(f"p-valor MC = {pValueMc:.4f} | p-valor teórico = {pValueTheo:.4f}")
ax.legend()


## Problema 7

Prueba de Kolmogorov-Smirnov para comparar salarios en 1989 y 1990.

In [ ]:
dataPath = Path("Problemas/Snmesp.csv")
if not dataPath.exists():
    dataPath = Path("Snmesp.csv")

salaryData = pd.read_csv(dataPath)
salary1989 = salaryData.loc[salaryData["year"] == 1989, "salary"].to_numpy()
salary1990 = salaryData.loc[salaryData["year"] == 1990, "salary"].to_numpy()

ksResult = stats.ks_2samp(salary1989, salary1990, alternative="two-sided", mode="auto")

sorted1989 = np.sort(salary1989)
sorted1990 = np.sort(salary1990)

cdf1989 = np.arange(1, sorted1989.size + 1) / sorted1989.size
cdf1990 = np.arange(1, sorted1990.size + 1) / sorted1990.size

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.step(sorted1989, cdf1989, where="post", label="1989")
ax.step(sorted1990, cdf1990, where="post", label="1990")
ax.set_xlabel("salario")
ax.set_ylabel("F(x)")
ax.set_title(f"KS estadístico = {ksResult.statistic:.3f} | p-valor = {ksResult.pvalue:.4f}")
ax.legend()
